# 02 · A network: the synfire chain

Connect groups of spiking neurons into a network that passes a wave of spikes from group to group, explore when the wave survives, and count the work it causes on the chip. Everything runs on the Brian2 simulator; participants with cluster access run the network on the SpiNNaker2 chip in section 3. Builds on notebook 01 (the `lif` neuron, weights and delays).

Work in pairs. **Try** marks exercises for the pair, **Discuss** marks questions we take up together.

This notebook is a short path through the vendor's tutorials, tailored to the workshop. Each section ends with **Go deeper**: where the content comes from and where to read more.

- Vendor tutorials (SpiNNcloud, Apache-2.0): [py-spinnaker2 `tutorials/deep_dive`](https://gitlab.com/spinnaker2/py-spinnaker2/-/tree/v0.8.2/tutorials/deep_dive), referred to as *tutorial 00*, *01*, …
- API documentation: [spinnaker2.gitlab.io/py-spinnaker2](https://spinnaker2.gitlab.io/py-spinnaker2/)
- **Workshop notes:** the folder [`01_knowledge_base`](../01_knowledge_base/) collects background written while preparing this workshop (spiking networks, the hardware, energy), with figures and further links. It is a reading collection for anyone who wants to delve deeper, not a primary source.

Collapsed blocks (▶) hold background for anyone who wants more; the exercises don't need them.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from spinnaker2 import brian2_sim, helpers, snn


def backend(hardware=False):
    """A fresh backend for every run: the Brian2 simulator, or one 48-chip SpiNNaker2 board."""
    if hardware:
        from spinnaker2 import hardware as hw

        return hw.SpiNNcloud48NodeBoard()
    return brian2_sim.Brian2Backend()

**Go deeper:** a network consists of four building blocks: populations, projections, a network and a backend. Brian2 and the chip share the same `run(net, steps)` method, so the code is written once (tutorial 00, sections 3–4).

## 1. A wave of spikes

A [synfire chain](https://en.wikipedia.org/wiki/Synfire_chain) is a chain of neuron groups, each group connected to the next, like the layers of a feed-forward network. A short burst of input to the first group starts a **volley**: a wave of spikes that travels from group to group.

<details><summary>Where synfire chains come from</summary>

Moshe Abeles proposed synfire chains in the 1980s as a way for the cortex to pass on precisely timed spikes. [Diesmann, Gewaltig and Aertsen (1999)](https://doi.org/10.1038/990101) showed in simulations that a volley either settles into a stable, sharp packet or dies out, depending on how many neurons take part and how synchronous they are, which is what the exercises below explore. Synfire chains are also a common example for spiking simulators and neuromorphic hardware, py-spinnaker2 included: the result is easy to check by eye and sensitive to errors in timing.

</details>

Here: 5 groups of 50 `lif` neurons. Every neuron gets 15 connections from random neurons of the previous group, with weight 4 and delay 2. The threshold is 30, so a neuron needs about 8 input spikes arriving close together to fire. The input is a "kick" of 50 neurons that fire once each, around time step 10, with some random jitter.

`run_chain` builds and runs the network; its arguments are the parameters to explore. It prints, per group, how many spikes there were and over how many time steps they were spread.

In [ ]:
def run_chain(weight=4, threshold=30, alpha_decay=0.8, kick_spread=2, n_groups=5, hardware=False, plot=True):
    """Build and run the synfire chain; returns the spikes of every group."""
    n, fan_in, delay, steps = 50, 15, 2, 40
    rng = np.random.default_rng(seed=1)  # fixed seed: same network and input on every run

    kick_times = rng.normal(10, kick_spread, n).round().clip(1)  # one spike per kick neuron, around step 10
    kick = snn.Population(size=n, neuron_model="spike_list", params={i: [int(t)] for i, t in enumerate(kick_times)}, name="kick")
    groups = [
        snn.Population(
            size=n,
            neuron_model="lif",
            params={"threshold": threshold, "alpha_decay": alpha_decay, "reset": "reset_to_v_reset", "v_reset": 0.0},
            name=f"group_{g}",
            record=["spikes"],
        )
        for g in range(n_groups)
    ]
    net = snn.Network("synfire_chain")
    net.add(kick, *groups)
    for pre, post in zip([kick] + groups[:-1], groups):  # kick -> group 0 -> group 1 -> ...
        conns = [[int(i), j, weight, delay] for j in range(n) for i in rng.permutation(n)[:fan_in]]
        net.add(snn.Projection(pre=pre, post=post, connections=conns))
    backend(hardware).run(net, steps)

    spikes = [g.get_spikes() for g in groups]
    for g, s in enumerate(spikes):
        times = [t for ts in s.values() for t in ts]
        spread = f"steps {min(times)}-{max(times)}" if times else "-"
        print(f"group {g}: {len(times):4d} spikes, {spread}")
    # every spike reaches on average fan_in neurons of the next group: one synaptic event each
    n_spikes = sum(len(ts) for s in spikes[:-1] for ts in s.values()) + n
    print(f"synaptic events: {n_spikes * fan_in}")

    if plot:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(kick_times, np.arange(n) - n, "|", color="0.6", ms=3)
        for g, s in enumerate(spikes):
            idx, t = helpers.spike_times_dict_to_arrays(s)
            ax.plot(t, np.asarray(idx) + g * n, "|", color="black", ms=3)
        ax.set_yticks([(g - 0.5) * n for g in range(n_groups + 1)], ["kick"] + [f"group {g}" for g in range(n_groups)])
        ax.set(xlim=(0, steps), xlabel="time step (1 step = 1 ms)", ylabel="neurons")
        plt.show()
    return spikes


chain_spikes = run_chain()

Each group fires 3 steps after the previous one (delay 2, plus 1). The kick is spread over several steps, but the volley gets sharper from group to group: by group 2, all 50 neurons fire in the same time step.

<details><summary>Why the volley gets sharper</summary>

A neuron fires as soon as enough of its 15 inputs have arrived, and it fires only once per volley (reset to 0, the inputs are used up). Every neuron of the next group samples 15 of the 50 spike times, and the early part of the volley already brings most neurons close to the threshold. The spread of the output is therefore smaller than the spread of the input, and after two or three groups all neurons fire in the same step. With too few or too weak inputs, the same mechanism works the other way: fewer neurons reach the threshold, the next group gets even less input, and the volley dies out.

</details>

<details><summary>How the synaptic events are counted</summary>

Every spike of a neuron is delivered to each of its outgoing connections: one synaptic event per connection. Each group has 50 × 15 = 750 incoming connections from 50 neurons, so a neuron has 15 outgoing connections on average. Events = (spikes of the kick and of groups 0–3) × 15; the spikes of the last group go nowhere. On SpiNNaker2, each event costs the receiving core a memory lookup and an addition, so the event count is a proxy for the work done. Whether that shows up in the power reading depends on the size: on our board, 100 active cores are invisible, 1,000 add a few watts (energy demo at the end of the workshop).

</details>

**Try:** call `run_chain` with one changed argument each time, e.g. `run_chain(weight=3)`.

1. `weight=3`: why does the wave die out, although 15 × 3 = 45 is still above the threshold of 30?
2. `kick_spread=4`: the same input spikes, only less synchronous. What happens?
3. `alpha_decay=0.5`: a stronger leak. What happens, and how is this related to exercise 2?
4. `threshold=20`: how often does each neuron fire now, and what happens to the number of synaptic events?

**Discuss:**

- The neurons act as *coincidence detectors*: they fire only if their inputs arrive together. Where is the information in this network: in which neurons fire, or in when they fire?
- On SpiNNaker2, the work, and so the energy, grows with the number of synaptic events, not with the number of neurons (exercise 4). What does that mean for designing networks for this chip?

**Go deeper:** tutorial 00, section 6 runs a larger synfire chain (1,250 neurons, 100,000 synapses) with the `lif_curr_exp` model, local inhibition and an animation of the wave. Tutorial 02, section 3 shows other wiring patterns (one-to-one, all-to-all, random, fixed fan-in), section 4 turns a weight matrix into a connection list, section 5 combines projections. Workshop notes: [why spikes can save energy](../01_knowledge_base/01_primer_snn_basics.md#9-why-this-can-save-energy) and [how spikes travel as packets on the chip](../01_knowledge_base/02_hardware_energy.md#4-communication-spikes-as-packets).

## 2. What you built

Over notebooks 01 and 02:

- `snn.Population`: a group of neurons of one model (`lif`) or an input (`spike_list`)
- `snn.Projection`: the connections between two populations, as a list of `[pre, post, weight, delay]`
- `snn.Network`: everything that runs together; `run(net, steps)` simulates it

In notebooks 03 and 04, the NIR import builds exactly these objects automatically from a PyTorch model.

**Go deeper:** the other neuron models (adaptive threshold, Izhikevich, current-based synapses, …) and their weight and delay ranges: tutorial 01, section 2 and the [API docs, neuron models](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/neuron-models.html). All classes and arguments: [API docs, snn](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/snn-api.html). When a run fails or a network doesn't fit on the chip: tutorial 05. Workshop notes: [other neuron models on the chip](../01_knowledge_base/01_primer_snn_basics.md#5-other-neuron-models-on-the-chip) and [when a network doesn't fit](../01_knowledge_base/02_hardware_energy.md#7-limits-in-practice-when-a-network-doesnt-fit).

## 3. Run the network on the SpiNNaker2 chip (cluster participants)

Set `USE_HARDWARE = True` below and run the cell. It runs the synfire chain with the default parameters on one 48-chip board and compares the spikes with Brian2. A chip run takes about one minute, mostly for powering up and booting the board; if all boards are busy, the run waits automatically.

Participants without cluster access: read along, the expected result is shown in the workshop.

**Go deeper:** choosing a board and how a network is placed on the cores: tutorial 04. The chip itself: tutorial 06. Workshop notes: [the chip at a glance](../01_knowledge_base/02_hardware_energy.md#1-the-chip-at-a-glance).

In [ ]:
USE_HARDWARE = False  # set to True on the cluster

print("Brian2:")
reference = run_chain(plot=False)
print("\nSpiNNaker2:" if USE_HARDWARE else "\nBrian2 again (USE_HARDWARE is False):")
result = run_chain(hardware=USE_HARDWARE)

same = all(
    {i: list(ts) for i, ts in a.items() if len(ts)} == {i: list(ts) for i, ts in b.items() if len(ts)}
    for a, b in zip(reference, result)
)
print("identical spike times on both backends" if same else "spike times differ")